In [1]:
import polars as pl
import ast
import numpy as np
import time
import os
from tqdm import tqdm
from multiprocessing import Pool
import itertools
import matplotlib.pyplot as plt
import gc
from collections import Counter
import math
import keras
from scipy import sparse
from keras import models, layers
import keras

In [ ]:
class CustomDataGenerator(keras.utils.Sequence):
    def __init__(self,
                src_data,
                batch_size, 
                dataset_type,
                shuffle=True,
                **kwargs):
        super().__init__(**kwargs)
        self.src_data = src_data
        self.batch_size = batch_size
        assert dataset_type in ["train", "validation", "test"]
        self.dataset_type = dataset_type
        self.shuffle = shuffle
        self.dataset = self.read_dataset()
        print(f"The length of dataset {self.dataset_type} is {self.get_n_row_dataset():,}")

    def get_n_row_dataset(self):
        return self.dataset.select(pl.len()).collect().item()
    
    def read_dataset(self):
        dataset_type_folder = os.path.join(self.src_data, f"data_{self.dataset_type}_folder")
        dataset = pl.scan_parquet(os.path.join(dataset_type_folder, "*.parquet"))
        return dataset

    def on_epoch_end(self):
        pass

    def __len__(self):
        return math.ceil(self.get_n_row_dataset() / self.batch_size)

    def normalize_dataset(self, dataset: pl.DataFrame):
        bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
        dataset = dataset.with_columns(
            [
                pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
                (pl.col("player2_action_max_card") + 1) / 52,  # Normalize player2_action_max_card
                (pl.col("player3_action_max_card") + 1) / 52,  # Normalize player3_action_max_card
                (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
            ]
        )
        return dataset

    def __getitem__(self, index):
        batch_data = self.dataset.slice(index * self.batch_size, self.batch_size).collect()
        batch_data = self.normalize_dataset(batch_data)
        batch_data = batch_data.to_numpy()
        X = batch_data[:, :-1]
        y = batch_data[:, -1]
        return X, y


src_data = r"F:\dataset"
batch_size = 4096
data_train_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="train")
data_validation_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="validation")

In [ ]:
model = models.Sequential()
model.add(layers.Input(shape=(data_train_generator[0][0].shape[1],)))
model.add(layers.Dense(1024, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(512, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(256, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(128, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(64, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(32, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(16, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(1, activation="sigmoid"))

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=r"TLMN_see_opponent_card_3_players.keras",
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1)

history = model.fit(data_train_generator, 
                    validation_data=data_validation_generator,
                    epochs=50)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Mean Squared Error')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Evaluate the model on the testing set
from keras.models import load_model

best_model = load_model(r"F:\Epoch_3_TLMN_see_opponent_card_20k_games_NVIDIA_Quadro_P2200.keras")
data_test_generator = CustomDataGenerator(
    src_data=r"F:\big_data\data_test", 
    dataset_type="test")
best_model.evaluate(data_test_generator)